In [130]:
import re
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

In [93]:
df = pd.read_csv("train.tsv",
                 sep="\t",
                 header=None,
                 names=["labels", "text"])

In [94]:
df["labels"] = df["labels"].map(str.split)

# Train NER

In [108]:
# Named entity recognition pipeline, passing in a specific model and tokenizer
model = AutoModelForTokenClassification.from_pretrained("dbmdz/bert-large-cased-finetuned-conll03-english")
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
recognizer = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [152]:
def split_prediction(predictions, idx):
    prediction = predictions[idx]
    entity_group = prediction['entity_group']
    words = prediction["word"].split()
    res = [f"B-{entity_group}"] + [f"I-{entity_group}"] * (len(words) - 1)
    return [list(el) for el in zip(words, res)]

In [154]:
def predict(text):
    predictions = recognizer(text)
    pred_labels = []
    for i in range(len(predictions)):
        word_label_pairs = split_prediction(predictions, i)
        if word_label_pairs[0][0].startswith("#"):
            pass
            # if pred_labels:
            #     pred_labels[-1][0] += re.sub("#+", "", word_label_pairs[0][0])
            # else:
            #     word_label_pairs[0][0] = text[:predictions[i]["end"]]
        else:
            pred_labels.extend(word_label_pairs)

    words = text.split()
    res = []
    pred_idx = 0
    for i, word in enumerate(words):
        if pred_idx < len(pred_labels) and word.startswith(pred_labels[pred_idx][0]):
            res.append(pred_labels[pred_idx][1])
            pred_idx += 1
        else:
            res.append("O") # outside of prediction (non-NER)
    return res

predict(df["text"][0])

['B-ORG',
 'O',
 'B-MISC',
 'O',
 'O',
 'O',
 'B-MISC',
 'O',
 'O',
 'O',
 'B-PER',
 'I-PER',
 'O',
 'B-LOC',
 'O',
 'O',
 'O',
 'B-ORG',
 'I-ORG',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-MISC',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-MISC',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-LOC',
 'O',
 'O',
 'O',
 'O',
 'B-ORG',
 'I-ORG',
 'O',
 'O',
 'O',
 'B-PER',
 'I-PER',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-LOC',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-ORG',
 'O',
 'O',
 'O',
 'B-PER',
 'I-PER',
 'I-PER',
 'I-PER',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-ORG',
 'I-ORG',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-ORG',
 'O',
 'O',
 'B-PE

# Evaluation

In [135]:
from seqeval.metrics import classification_report

In [136]:
def get_scores(y_true, y_pred):
    # Funkcja zwraca precyzję, pokrycie i F1
    acc_score = 0
    tp = 0
    fp = 0
    selected_items = 0
    relevant_items = 0

    for p, t in zip(y_pred, y_true):
        if p == t:
            acc_score += 1

        if p > 0 and p == t:
            tp += 1

        if p > 0:
            selected_items += 1

        if t > 0:
            relevant_items += 1

    if selected_items == 0:
        precision = 1.0
    else:
        precision = tp / selected_items

    if relevant_items == 0:
        recall = 1.0
    else:
        recall = tp / relevant_items

    if precision + recall == 0.0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

In [155]:
predictions = []
i = 0
for text in df["text"]:
    try:
        predictions.append(predict(text))
        i += 1
    except Exception as e:
        print(f"failed {i}: error: {e}")

In [156]:
print(classification_report(df["labels"], predictions))

              precision    recall  f1-score   support

         LOC       0.96      0.54      0.69      7139
        MISC       0.88      0.60      0.72      3436
         ORG       0.93      0.54      0.69      6317
         PER       0.96      0.44      0.60      6600

   micro avg       0.94      0.52      0.67     23492
   macro avg       0.93      0.53      0.67     23492
weighted avg       0.94      0.52      0.67     23492

